In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1: Sistem Siap!")

Cell 1: Sistem Siap!


In [2]:
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

def align_data(df):
    # Ekstrak angka dari battery
    df['Battery_Level'] = df['Battery_Level'].astype(str).str.extract('(\d+)').astype(float)
    # Standarisasi Signal
    df['Signal_Strength'] = df['Signal_Strength'].astype(str).apply(
        lambda x: 'Low' if x in ['Edge', '3G', 'None'] else 'High'
    )
    return reduce_mem(df)

df_train = align_data(df_train)
df_test = align_data(df_test)
print("Cell 2: Data Ter-sinkronisasi!")

Cell 2: Data Ter-sinkronisasi!


In [3]:
def build_final_features(df):
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # 1. Konteks Waktu: Malam hari rawan Safety/Fraud
    df['is_night'] = df['hour'].apply(lambda x: 1 if x >= 23 or x <= 4 else 0)
    
    # 2. Tech Stress: Sinyal buruk + GPS tidak akurat
    df['nav_stress'] = ((df['GPS_Accuracy_M'] > 20) & (df['Signal_Strength'] == 'Low')).astype(int)
    
    # 3. Geofencing Ringan: Jarak dari pusat (Asumsi Jakarta/Pusat Aktivitas)
    avg_lat, avg_long = -6.2, 106.8
    df['dist_from_center'] = np.sqrt((df['Pickup_Lat']-avg_lat)**2 + (df['Pickup_Long']-avg_long)**2)
    
    # 4. Rute & Harga: Deteksi anomali ekonomi
    lat_diff = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    long_diff = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    euclidean = np.sqrt(lat_diff**2 + long_diff**2)
    df['route_anomaly'] = df['Distance_KM'] / (euclidean + 0.005)
    df['price_per_km'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 1)
    
    # 5. Sensor Magnitude
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    
    # Pilih hanya kolom numerik murni + Signal_Strength
    cols_to_keep = [
        'hour', 'is_night', 'nav_stress', 'dist_from_center', 
        'route_anomaly', 'accel_mag', 'price_per_km',
        'Distance_KM', 'Est_Price_IDR', 'GPS_Accuracy_M',
        'Signal_Strength', 'Battery_Level'
    ]
    return df[cols_to_keep]

y = df_train['Trip_Label']
X = build_final_features(df_train)
X_test_final = build_final_features(df_test)

del df_train
gc.collect()
print("Cell 3: Fitur Cerdas Selesai!")

Cell 3: Fitur Cerdas Selesai!


In [4]:
# Ambil 10% agar RAM tetap dingin saat proses validasi
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.10, stratify=y, random_state=42)

del X, y
gc.collect()

cat_features = ['Signal_Strength']
for col in cat_features:
    X_train[col] = X_train[col].fillna("UNK").astype(str)
    X_val[col] = X_val[col].fillna("UNK").astype(str)
    X_test_final[col] = X_test_final[col].fillna("UNK").astype(str)

print(f"Cell 4: Validasi siap. Baris Train: {len(X_train)}")

Cell 4: Validasi siap. Baris Train: 800000


/tmp/ipykernel_177931/2116729158.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test_final[col] = X_test_final[col].fillna("UNK").astype(str)


In [5]:
model = CatBoostClassifier(
    iterations=600,
    learning_rate=0.1,
    depth=5, 
    task_type="CPU",
    thread_count=10,
    loss_function='MultiClass',
    # Memberikan bobot tinggi pada kelas sulit
    class_weights={'Perfect_Trip': 1, 'Safety_Violation': 2, 'Fraud_Indication': 4, 
                   'Navigation_Issue': 8, 'Service_Complaint': 8},
    early_stopping_rounds=30,
    random_seed=42,
    verbose=100
)

model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 1.5315793	test: 1.5316780	best: 1.5316780 (0)	total: 443ms	remaining: 4m 25s
100:	learn: 1.1216002	test: 1.1231210	best: 1.1231210 (100)	total: 27.6s	remaining: 2m 16s
200:	learn: 1.1179132	test: 1.1209663	best: 1.1209652 (198)	total: 53.5s	remaining: 1m 46s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 1.120566773
bestIteration = 268

Shrink model to first 269 iterations.


In [9]:
probs = model.predict_proba(X_val)
class_names = model.classes_
prob_df = pd.DataFrame(probs, columns=class_names)

# --- REVISI MULTIPLIER (TITIK TENGAH) ---
# Kita turunkan pengali minoritas agar tidak terlalu mendominasi
prob_df['Navigation_Issue'] *= 1.4    # Turun dari 2.0
prob_df['Service_Complaint'] *= 1.4   # Turun dari 2.0
prob_df['Fraud_Indication'] *= 1.1    # Turun dari 1.2
prob_df['Safety_Violation'] *= 1.0    # Baseline
prob_df['Perfect_Trip'] *= 1.5        # KITA NAIKKAN INI AGAR TIDAK 0.00

y_val_pred = prob_df.idxmax(axis=1)

print("HASIL EVALUASI REVISI (PENYELAMATAN):")
f1 = f1_score(y_val, y_val_pred, average='macro')
print(f"Macro F1 Score: {f1:.4f}")
print("-" * 50)
print(classification_report(y_val, y_val_pred))

HASIL EVALUASI REVISI (PENYELAMATAN):
Macro F1 Score: 0.3804
--------------------------------------------------
                   precision    recall  f1-score   support

 Fraud_Indication       0.91      0.97      0.94    360282
 Navigation_Issue       0.12      0.53      0.19    721611
     Perfect_Trip       0.64      0.00      0.00   3957846
 Safety_Violation       0.87      0.44      0.59   1441436
Service_Complaint       0.11      0.45      0.18    718825

         accuracy                           0.24   7200000
        macro avg       0.53      0.48      0.38   7200000
     weighted avg       0.60      0.24      0.20   7200000



In [7]:
test_probs = model.predict_proba(X_test_final)
test_prob_df = pd.DataFrame(test_probs, columns=class_names)

# Terapkan pengali yang sama dengan Cell 6
test_prob_df['Navigation_Issue'] *= 2.0
test_prob_df['Service_Complaint'] *= 2.0
test_prob_df['Fraud_Indication'] *= 1.2
test_prob_df['Safety_Violation'] *= 1.1

test_preds = test_prob_df.idxmax(axis=1)

submission = pd.DataFrame({"Trip_ID": df_test["Trip_ID"], "Trip_Label": test_preds})
submission.to_csv(os.path.join(SUB_PATH, "submission_final_06_target.csv"), index=False)
print("Bismillah! Peluru terakhir sudah siap. Silakan di-submit!")

Bismillah! Peluru terakhir sudah siap. Silakan di-submit!
